# Executor Scaffold Pipeline

Shared walk-forward backtest scaffold for the four ML executors
(`ml_xgboost`, `ml_lightgbm`, `ml_random_forest`, `ml_ridge`). Each
per-method script in `src/ml_*.py` calls into this module so that
* CLI parsing,
* data loading + feature generation,
* horizon shifting + chunk slicing,
* walk-forward backtest + Duan smearing + chunk reduce JSON

are maintained in exactly one place. The per-method script supplies
only:
1. a `fit_predict(X_chunk, y_chunk, train_win_periods, hyperparams)`
   callable (which internally uses `src.scaling.run_backtest` with the
   right `use_scaling` / `refit_frequency`),
2. a default-hyperparams dict,
3. and (for Ridge only) the segment / lag-scope branching.

The `run_executor` function is the canonical entry point. The CLI
contract (`--data-path`, `--horizon`, `--train-window`, `--start`,
`--end`, `--exog-cols`, `--params-file`, `--output-file`,
`--refit-frequency`, `--segment`, `--lag-scope`, `--seed`) is owned
by `parse_executor_args()`.

The final cell exports `../../src/executor.py`.

In [ ]:
# ── Setup: clone repo, install deps ──
import os
import sys

REPO_URL = "https://github.com/jamesdchen/harxhar.git"
REPO_DIR = "/content/harxhar"
if not os.path.isdir(REPO_DIR):
    !git clone {REPO_URL} {REPO_DIR}
os.chdir(REPO_DIR)
!pip install -q numpy pandas pyarrow numba scikit-learn tqdm

sys.path.insert(0, "/content/harxhar")

## Imports + module docstring

First export cell: module docstring and the imports the rest of the file
leans on. Helpers come from `src.loading`, `src.transforms`,
`src.scaling`, `src.evaluation` — nothing is reimplemented here.

In [ ]:
# export
"""Shared executor scaffold for ML walk-forward volatility backtests.

Lifts the ~90% duplicated `main()` body of the per-method executors
(`src/ml_xgboost.py`, `src/ml_lightgbm.py`, `src/ml_random_forest.py`,
`src/ml_ridge.py`) into one module. Per-method scripts supply only a
`fit_predict(X_chunk, y_chunk, train_win_periods, hyperparams)`
callable plus a default-hyperparam dict; everything else — CLI parsing,
loading, transforms, horizon shift, chunk slicing, smearing, reduce
JSON — is shared.

The CLI contract is now owned by ``.hpc/cli.py`` (the auto-generated
dispatcher) and ``.hpc/tasks.py`` FLAGS dict — the per-executor flag
list lives there, not in this module. ``run_executor`` and the
backtest plumbing remain here. The flags expected on ``args`` match
those declared in ``.hpc/tasks.py`` FLAGS for the calling executor's
module key (e.g. ``FLAGS["src/tune_tree.py`` (cmd_evaluate) passes via subprocess:
``--params-file --output-file --start --end --data-path --train-window``
``--horizon --exog-cols`` plus the Ridge-only ``--segment`` and
``--lag-scope`` and the optional ``--refit-frequency`` / ``--seed``.
"""

from __future__ import annotations

import os
from collections.abc import Callable

import numpy as np
import pandas as pd

from src.evaluation import apply_duan_smearing, save_chunk_reduce
from src.loading import apply_overnight_fills, load_raw_data
from src.transforms import (
    PERIODS_PER_DAY,
    SEGMENT_DEFINITIONS,
    add_calendar_features,
    apply_horizon_shift,
    compute_segment_train_window,
    generate_har_features,
    resolve_har_lags,
    robust_transform,
    slice_to_segment,
)

FitPredict = Callable[[np.ndarray, np.ndarray, int, dict], np.ndarray]

In [ ]:
# ── Sanity check: every symbol resolved ──
_expected = [
    "argparse",
    "os",
    "np",
    "pd",
    "apply_duan_smearing",
    "save_chunk_reduce",
    "apply_overnight_fills",
    "load_raw_data",
    "PERIODS_PER_DAY",
    "SEGMENT_CHOICES",
    "SEGMENT_DEFINITIONS",
    "add_calendar_features",
    "apply_horizon_shift",
    "compute_segment_train_window",
    "generate_har_features",
    "resolve_har_lags",
    "robust_transform",
    "slice_to_segment",
    "FitPredict",
]
_missing = [n for n in _expected if n not in dir()]
assert not _missing, f"missing imports: {_missing}"
print(f"imports OK ({len(_expected)} symbols)")
print(f"PERIODS_PER_DAY={PERIODS_PER_DAY}, SEGMENT_CHOICES={SEGMENT_CHOICES}")

In [ ]:
# export
from dataclasses import dataclass  # noqa: E402


@dataclass(frozen=True)
class ExecutorConfig:
    """Per-method backtest invariants. Drift here was the proximate cause
    of the 2026-04-23 alignment audit (intersection-N regression). Each
    per-method module (``src/ml_<method>.py``) defines its own
    ``CONFIG = ExecutorConfig(...)`` constant; the CONFIGS registry
    below imports them at runtime for the drift-check inside
    ``run_executor``.
    """

    method: str
    add_calendar: bool
    target_use_diurnal: bool
    target_winsor_window: int | None
    dropna_with_exog: bool
    refit_frequency: int

    def as_data_prep_kwargs(self) -> dict:
        return {
            "add_calendar": self.add_calendar,
            "target_use_diurnal": self.target_use_diurnal,
            "target_winsor_window": self.target_winsor_window,
            "dropna_with_exog": self.dropna_with_exog,
        }

In [ ]:
# export
def _load_configs() -> dict[str, ExecutorConfig]:
    """Lazy registry — imports CONFIG from each method module so per-method
    specs live next to per-method model code (not centralized here).

    Lazy because of the circular import: ``src.ml_<method>`` imports
    ``ExecutorConfig`` from this module, so we can't import them at
    module-import time. ``run_executor`` calls this once when needed.

    Tolerant by design: a method module migrated to the ``@register_run``
    experiment template no longer defines ``CONFIG`` (its data-prep
    invariants are inline). Such modules are skipped so the registry — and
    the drift-check it backs — keeps working through an incremental
    migration.

    Excluded by design: dl_ae_ridge, dl_patchts (separate config shape);
    tune_tree (no run_executor call); ml_baseline (no exog path).
    """
    import importlib

    out: dict[str, ExecutorConfig] = {}
    for modname in (
        "src.ml_ridge",
        "src.ml_xgboost",
        "src.ml_lightgbm",
        "src.ml_random_forest",
        "src.ml_pcr",
    ):
        try:
            cfg = importlib.import_module(modname).CONFIG
        except (ImportError, AttributeError):
            continue
        out[cfg.method] = cfg
    return out


# Lazy singleton — populated on first access via _get_configs(); CONFIGS
# is exported as a module-level name so existing call sites that import
# CONFIGS keep working unchanged.
_CONFIGS_CACHE: dict[str, ExecutorConfig] | None = None


def _get_configs() -> dict[str, ExecutorConfig]:
    global _CONFIGS_CACHE
    if _CONFIGS_CACHE is None:
        _CONFIGS_CACHE = _load_configs()
    return _CONFIGS_CACHE


class _ConfigsProxy:
    """Dict-like proxy that delegates to the lazy-loaded registry."""

    def __getitem__(self, k):
        return _get_configs()[k]

    def __contains__(self, k):
        return k in _get_configs()

    def __iter__(self):
        return iter(_get_configs())

    def keys(self):
        return _get_configs().keys()

    def items(self):
        return _get_configs().items()

    def values(self):
        return _get_configs().values()


CONFIGS = _ConfigsProxy()

## `parse_executor_args` — canonical CLI parser

Owns ALL flags that `src/tune_tree.py` passes via
`subprocess.run([sys.executable, "src/ml_<method>.py", ...])`:

| flag | type | default | notes |
|------|------|---------|-------|
| `--data-path` | str | `"all30min"` | passed by tuner |
| `--horizon` | int | 1 | passed by tuner |
| `--train-window` | int | 500 | days; passed by tuner |
| `--start` | int | 0 | passed by tuner |
| `--end` | int | -1 | passed by tuner |
| `--exog-cols` | str | None | pipe-separated; passed by tuner |
| `--output-file` | str | required | passed by tuner |
| `--params-file` | str | None | passed by tuner |
| `--refit-frequency` | int | None | sentinel — falls back to per-method default if not set |
| `--segment` | str | None | choices: `SEGMENT_CHOICES`; Ridge only |
| `--lag-scope` | str | `"global"` | choices: `["global", "intra"]`; Ridge only |
| `--seed` | int | 42 | reserved for future use |

The `--refit-frequency` sentinel of `None` lets each method keep its
original hardcoded default (XGB=1, LGBM=1, RF=5, Ridge=1) without the
tuner having to know per-method defaults.

## `_backtest_and_save` — lifted scaffold body

The core walk-forward block: horizon shift, chunk slice, train-window
guard, `fit_predict` call (which the per-method script wires to
`run_backtest` with the right scaling/refit settings), Duan smearing,
CSV write, reduce JSON.

This is a verbatim lift of `src/ml_ridge.py:_run_backtest_and_save`,
with two changes:
1. The `Ridge(alpha=1.0)` model factory is replaced by a generic
   `fit_predict` callable.
2. `dropna` happens upstream (in `run_executor`), not here.

In [ ]:
# export
def _backtest_and_save(
    df: pd.DataFrame,
    feature_names: list[str],
    fit_predict: FitPredict,
    hyperparams: dict,
    train_win_periods: int,
    horizon: int,
    start: int,
    end: int,
    output_file: str,
) -> None:
    """Run a prepared DataFrame through the walk-forward backtest and save.

    Parameters
    ----------
    df : pd.DataFrame
        Must contain ``adj_RV``, ``baseline``, ``t``, plus all
        ``feature_names`` columns. The HAR-burn-in (``max_lag``) rows
        are dropped here before any extraction.
    feature_names : list[str]
        Column names in ``df`` that form the feature matrix.
    fit_predict : FitPredict
        Callable ``(X_chunk, y_chunk, train_win_periods, hyperparams)
        -> preds`` of shape ``(len(X_chunk) - train_win_periods,)``.
    hyperparams : dict
        Method-specific hyperparameters (typically a merge of method
        defaults and tuned overrides from ``--params-file``).
    train_win_periods : int
        Training window in 30-min periods.
    horizon : int
        Forecast horizon in 30-min periods.
    start, end : int
        Inclusive/exclusive chunk bounds in *post horizon-shift* index
        space. ``end == -1`` means "to the end".
    output_file : str
        Output CSV path. ``<basename>_reduce.json`` is written
        alongside.
    """
    max_lag = resolve_har_lags()[-1]
    df = df.iloc[max_lag:].reset_index(drop=True)

    X = df[feature_names].values.astype(np.float64)
    y = df["adj_RV"].values.astype(np.float64)
    dates = df["t"]
    baselines = df["baseline"].values.astype(np.float64)

    X, y, dates, baselines = apply_horizon_shift(X, y, dates, baselines, horizon)

    actual_end = len(X) if end == -1 else end
    X_chunk = X[start:actual_end]
    y_chunk = y[start:actual_end]
    dates_chunk = dates.iloc[start:actual_end].reset_index(drop=True)
    baselines_chunk = baselines[start:actual_end]

    if train_win_periods >= len(X_chunk):
        raise ValueError(f"train_window ({train_win_periods} periods) >= chunk size ({len(X_chunk)})")

    preds = fit_predict(X_chunk, y_chunk, train_win_periods, hyperparams)

    oos_start = train_win_periods
    y_oos = y_chunk[oos_start:]
    dates_oos = dates_chunk.iloc[oos_start:].values
    baselines_oos = baselines_chunk[oos_start:]

    pred_raw, true_raw = apply_duan_smearing(preds, y_oos, baselines_oos)

    results = pd.DataFrame(
        {
            "date": dates_oos,
            "horizon": horizon,
            "true_adj": y_oos,
            "pred_adj": preds,
            "true_raw": true_raw,
            "pred_raw": pred_raw,
        }
    )

    os.makedirs(os.path.dirname(output_file) or ".", exist_ok=True)
    results.to_csv(output_file, index=False)
    save_chunk_reduce(results, output_file)
    print(f"Saved {len(results)} rows -> {output_file}")


def _build_har_and_calendar(df, exog_cols, add_calendar):
    df, har_names = generate_har_features(df, target_col="adj_RV", exog_cols=exog_cols)
    if add_calendar:
        feature_names = har_names + add_calendar_features(df)
    else:
        feature_names = har_names
    return df, feature_names

In [ ]:
# ── Smoke test: _backtest_and_save with a trivial model on synthetic data ──
import tempfile
from pathlib import Path as _Path

# Build a minimal synthetic df that has the columns _backtest_and_save expects.
rng = np.random.default_rng(0)
n = 800
idx = pd.date_range("2020-01-01", periods=n, freq="30min")
feat_names = ["f1", "f2", "f3"]
_df = pd.DataFrame(
    {
        "t": idx,
        "adj_RV": rng.normal(size=n),
        "baseline": np.ones(n),
        "f1": rng.normal(size=n),
        "f2": rng.normal(size=n),
        "f3": rng.normal(size=n),
    }
)


def _trivial_fp(X_chunk, y_chunk, train_win, hp):
    # constant predictor; no scaling/refit needed for the scaffold test
    return np.full(len(X_chunk) - train_win, hp.get("const", 0.5))


with tempfile.TemporaryDirectory() as tmp:
    out = os.path.join(tmp, "chunk.csv")
    _backtest_and_save(
        _df,
        feat_names,
        _trivial_fp,
        {"const": 0.5},
        train_win_periods=200,
        horizon=1,
        start=0,
        end=-1,
        output_file=out,
    )
    assert _Path(out).is_file()
    res = pd.read_csv(out)
    expected_cols = ["date", "horizon", "true_adj", "pred_adj", "true_raw", "pred_raw"]
    assert list(res.columns) == expected_cols, res.columns.tolist()
    assert _Path(out.replace(".csv", "_reduce.json")).is_file()
    print(f"_backtest_and_save OK: {len(res)} rows, columns={list(res.columns)}")

## `load_and_transform` — dataset prep helper

Shared load + diurnal/winsor RV transform + exog adjustment used by
all four methods. Per-method differences:
* `dropna_with_exog`: Ridge & RF drop rows with any NaN in `RV` *or*
  `exog_cols`; XGB/LGBM only drop on `RV` (XGBoost/LightGBM handle
  feature NaN natively).
* `use_diurnal_target` / `target_winsor_window`: Ridge runs the
  baseline RV transform with neither (default). Tree methods use
  `use_diurnal=True, winsor_window=240`.

These are the only data-prep knobs; everything else is uniform across
methods.

In [ ]:
# export
def load_and_transform(
    data_path: str,
    exog_cols: list[str],
    *,
    target_use_diurnal: bool,
    target_winsor_window: int | None,
    dropna_with_exog: bool,
) -> tuple[pd.DataFrame, list[str]]:
    """Load raw data, apply RV + exog robust transforms, return (df, adj_exog).

    Parameters
    ----------
    data_path : str
        Forwarded to ``load_raw_data``.
    exog_cols : list[str]
        Raw exog column names (pre-transform). May be empty.
    target_use_diurnal : bool
        If True, apply diurnal adjustment to the RV target transform.
        Tree methods: True. Ridge: False.
    target_winsor_window : int | None
        Winsorization window for the RV target. Tree methods: 240.
        Ridge: None (no winsorization beyond ``robust_transform``
        defaults).
    dropna_with_exog : bool
        Exog columns are forward-filled before this branch, so the only
        rows that can still have NaN exog are at the leading edge of the
        dataset (before any valid value). If True, drop those rows.
        Ridge: True (sklearn Ridge requires no NaN). Tree methods: False
        (trees handle NaN natively).
    """
    df = load_raw_data(data_path, allow_missing=True)
    if exog_cols:
        apply_overnight_fills(df, exog_cols)
        df[exog_cols] = df[exog_cols].ffill()
        if dropna_with_exog:
            df = df.dropna(subset=["RV"] + exog_cols).reset_index(drop=True)
        else:
            df = df.dropna(subset=["RV"]).reset_index(drop=True)

    transform_kwargs: dict = {"is_target": True}
    if target_use_diurnal:
        transform_kwargs["use_diurnal"] = True
    if target_winsor_window is not None:
        transform_kwargs["winsor_window"] = target_winsor_window
    adj_rv, baseline = robust_transform(df, "RV", **transform_kwargs)
    df["adj_RV"] = adj_rv
    df["baseline"] = baseline

    adj_exog_cols: list[str] = []
    for col in exog_cols:
        adj_col = f"adj_{col}"
        adj_series, _ = robust_transform(df, col, use_transform=True, use_diurnal=True)
        df[adj_col] = adj_series
        adj_exog_cols.append(adj_col)

    return df, adj_exog_cols

## `run_executor` — the top-level entry

Branches on `segment is None`:
* No segment: build features (HAR + optional calendar), run
  `_backtest_and_save` once.
* Segment given (Ridge only): loop over segments, with `lag_scope`
  controlling whether HAR features are computed once on the whole
  dataset (`global`) or per-segment (`intra`). Each segment writes to
  `<output>_<seg>.csv`.

The per-method script invokes this with method-specific
`fit_predict` + `hyperparams` + boolean toggles.

In [ ]:
# export
def _iter_TOD_segment(
    df,
    *,
    segment,
    lag_scope,
    train_window,
    output_file,
    exog_cols,
    add_calendar,
):
    """Yield (seg_name, job_df, feature_names, train_win_periods, job_output_file)
    for each time-of-day segment we need to backtest. ``seg_name`` is None when
    no segmentation is requested."""
    if segment is None:
        df, feature_names = _build_har_and_calendar(df, exog_cols, add_calendar)
        yield None, df, feature_names, train_window * PERIODS_PER_DAY, output_file
        return

    segments = list(SEGMENT_DEFINITIONS) if segment == "all" else [segment]
    base, ext = os.path.splitext(output_file)

    if lag_scope == "global":
        df, feature_names = _build_har_and_calendar(df, exog_cols, add_calendar)

    for seg_name in segments:
        seg_df = slice_to_segment(df, seg_name)
        if seg_df.empty:
            print(f"No data for segment '{seg_name}'. Skipping.")
            continue
        if lag_scope == "intra":
            seg_df, feature_names = _build_har_and_calendar(seg_df, exog_cols, add_calendar)
        train_win_periods = compute_segment_train_window(seg_df["t"], train_window)
        yield seg_name, seg_df, feature_names, train_win_periods, f"{base}_{seg_name}{ext}"


def run_executor(
    method_name: str,
    fit_predict: FitPredict,
    hyperparams: dict,
    *,
    data_path: str,
    output_file: str,
    horizon: int,
    train_window: int,
    start: int,
    end: int,
    exog_cols: list[str],
    segment: str | None,
    lag_scope: str,
    add_calendar: bool,
    target_use_diurnal: bool,
    target_winsor_window: int | None,
    dropna_with_exog: bool,
    seed: int = 42,
) -> None:
    """Top-level scaffold. Loads data, builds features, dispatches backtest.

    Per-method scripts (`ml_xgboost.py`, etc.) call this from their
    ``main()`` after parsing the canonical CLI and assembling the
    method-specific ``fit_predict`` closure + merged hyperparams.

    Notes
    -----
    * ``segment`` is None for tree methods (XGB/LGBM/RF). Ridge is the
      only method that uses segments.
    * ``add_calendar`` is True for tree methods, False for Ridge.
    * ``method_name`` is informational — used in log lines only.
    """
    del seed  # reserved; per-method scripts can wire seed into model_fn directly

    if method_name in CONFIGS:
        expected = CONFIGS[method_name].as_data_prep_kwargs()
        actual = {
            "add_calendar": add_calendar,
            "target_use_diurnal": target_use_diurnal,
            "target_winsor_window": target_winsor_window,
            "dropna_with_exog": dropna_with_exog,
        }
        if actual != expected:
            raise ValueError(f"data-prep drift for {method_name}: {actual} != {expected}")

    df, adj_exog_cols = load_and_transform(
        data_path,
        exog_cols,
        target_use_diurnal=target_use_diurnal,
        target_winsor_window=target_winsor_window,
        dropna_with_exog=dropna_with_exog,
    )

    for seg_name, job_df, feature_names, train_win, out_file in _iter_TOD_segment(
        df,
        segment=segment,
        lag_scope=lag_scope,
        train_window=train_window,
        output_file=output_file,
        exog_cols=adj_exog_cols,
        add_calendar=add_calendar,
    ):
        if seg_name is not None:
            print(f"{'=' * 20} {method_name.upper()} SEGMENT: {seg_name.upper()} {'=' * 20}")
            print(f"Window: {train_win} periods ({train_window} days)")
        _backtest_and_save(
            job_df,
            feature_names,
            fit_predict,
            hyperparams,
            train_win,
            horizon,
            start,
            end,
            out_file,
        )

In [ ]:
# ── Sanity: signatures bound, parser callable, no segment-without-lag-scope clash ──
import inspect

assert callable(run_executor)
assert callable(_backtest_and_save)
sig = inspect.signature(run_executor)
required_kw = {
    "data_path",
    "output_file",
    "horizon",
    "train_window",
    "start",
    "end",
    "exog_cols",
    "segment",
    "lag_scope",
    "add_calendar",
    "target_use_diurnal",
    "target_winsor_window",
    "dropna_with_exog",
}
actual = set(sig.parameters)
missing = required_kw - actual
assert not missing, f"run_executor missing kwargs: {missing}"
print("run_executor signature OK; kwargs:", sorted(actual))

## Re-export

Write the concatenated `# export` cells out to `../../src/executor.py`.

In [ ]:
import sys

sys.path.insert(0, "..")
from _exporter import export_notebook

export_notebook("05_executor.ipynb", "../../src/executor.py")